#  Evolution of Anime Production Over Time by Format

In this analysis, we will explore how the production of anime has evolved over the years, categorized by different formats such as TV series, movies, OVAs, etc. We will create a stacked area chart to visualize the trends in anime production over time.

In [1]:
import pandas as pd
import plotly.express as px
from lib import dbconnection


## 1. Query Data from PostgreSQL

In [2]:
# Estraiamo anche il 'type' (TV, Movie, OVA, ecc.) oltre alla data
engine = dbconnection.create_db_engine()
query = """
    SELECT start_date, type
    FROM details
    WHERE start_date IS NOT NULL AND type IS NOT NULL
"""
df = pd.read_sql(query, engine)


## 2. DATA CLEANING

In [3]:
# Convertiamo la data e gestiamo errori
df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
df = df.dropna(subset=['start_date']) # Rimuoviamo date non valide

# Estraiamo l'anno
df['year'] = df['start_date'].dt.year

# Filtriamo anni futuri (o errori tipo anno 2026/2030 se presenti e non voluti)
# e anni troppo vecchi se vuoi un focus moderno (opzionale)
df = df[df['year'] < 2026]


## 3. Data Aggregation

In [4]:
# Contiamo quanti anime per ogni Anno E per ogni Tipo
df_counts = df.groupby(['year', 'type']).size().reset_index(name='count')

# Filtriamo i tipi poco rilevanti se fanno rumore (es. "Music" o "Unknown")
df_counts = df_counts[df_counts['type'].isin(['TV', 'Movie', 'OVA', 'ONA', 'Special'])]

## 4. Visualisation (Stacked Area Chart)

In [5]:
fig = px.area(
    df_counts,
    x="year",
    y="count",
    color="type", # Questo crea la divisione per colori
    title="Evolution of Anime Production Over Time by Format",
    labels={"year": "Year", "count": "Number of released titles", "type": "Type"},
    template="plotly_white"
)

# Miglioriamo l'interattività
fig.update_layout(
    xaxis=dict(title="Year", rangeslider=dict(visible=True)),
    yaxis=dict(title="Production Count"),
    legend_title="Avg type"
)

# Salva o mostra
fig.write_html("../../graphs/production_evolution.html")
fig.show()